Dark Zurich 데이터 경로 탐색

In [ ]:
# Colab 새 셀
import os

# Dark Zurich 경로 찾기
!find /content -name "*labelTrainIds.png" 2>/dev/null | head -5
!find /content -type d -name "rgb_anon" 2>/dev/null
!find /content -type d -name "gt" 2>/dev/null

Google Drive 마운트 + Dark Zurich 압축 해제

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# zip 압축 해제 (경로는 너가 올린 위치 따라 수정)
!unzip -q "/content/drive/MyDrive/Dark_Zurich_val_anon.zip" -d /content/darkzurich/
!ls /content/darkzurich/

In [ ]:
# 압축 해제된 폴더 안 훑어보기
import os
for root, dirs, files in os.walk('/content/darkzurich'):
    level = root.replace('/content/darkzurich', '').count(os.sep)
    if level <= 3:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in files[:3]:
            print(f'{indent}  {f}')

라벨-이미지 개수 확인

In [ ]:
import glob
labels = glob.glob('/content/darkzurich/Dark_Zurich_val_anon/gt/val/**/*labelTrainIds.png', recursive=True)
rgbs = glob.glob('/content/darkzurich/Dark_Zurich_val_anon/rgb_anon/val/**/*rgb_anon.png', recursive=True)
print(f"📊 라벨: {len(labels)}장")
print(f"📷 이미지: {len(rgbs)}장")
print(f"\n샘플 라벨: {labels[0] if labels else '❌ 없음'}")
print(f"샘플 이미지: {rgbs[0] if rgbs else '❌ 없음'}")

In [ ]:
# ============================================================
# 📦 Dark Zurich → 우리 4클래스 변환 + 원본 200장 병합
# ============================================================
import os, glob, shutil
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm

# ---- 4클래스 정의 (SegFormer용) ----
# 0: Undrivable, 1: Road, 2: Lane Mark, 3: Background
# 움직이는 객체(차/사람/바이크)는 255(ignore) → YOLO가 담당
CLASS_NAMES_4 = ['Undrivable', 'Road', 'Lane Mark', 'Background']

# ---- Cityscapes 19 trainId → 우리 4클래스 ----
CS_TO_OURS = {
    0: 1,    # road        → Road
    1: 0,    # sidewalk    → Undrivable
    2: 0,    # building    → Undrivable
    3: 0,    # wall        → Undrivable
    4: 0,    # fence       → Undrivable
    5: 0,    # pole        → Undrivable
    6: 0,    # traffic light → Undrivable
    7: 0,    # traffic sign  → Undrivable
    8: 3,    # vegetation  → Background
    9: 3,    # terrain     → Background
    10: 3,   # sky         → Background
    11: 255, # person      → ignore (YOLO)
    12: 255, # rider       → ignore (YOLO)
    13: 255, # car         → ignore (YOLO)
    14: 255, # truck       → ignore (YOLO)
    15: 255, # bus         → ignore (YOLO)
    16: 255, # train       → ignore (YOLO)
    17: 255, # motorcycle  → ignore (YOLO)
    18: 255, # bicycle     → ignore (YOLO)
}

def cityscapes_to_ours4(label_np):
    """Cityscapes 19클래스 → 우리 4클래스"""
    out = np.full_like(label_np, 255, dtype=np.uint8)
    for cs_id, our_id in CS_TO_OURS.items():
        out[label_np == cs_id] = our_id
    # 원본 255(void)는 그대로 255
    out[label_np == 255] = 255
    return out


def extract_lane_marks(rgb_img, road_mask):
    """도로 영역에서 흰/노란 차선 추출 (색 기반 휴리스틱)"""
    hsv = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2HSV)
    # 야간이라 밝기 임계값 낮춤
    white = cv2.inRange(hsv, (0, 0, 160), (180, 40, 255))
    yellow = cv2.inRange(hsv, (15, 70, 100), (35, 255, 255))
    # 도로(1) 안에서만 차선 인정
    lane = ((white | yellow) > 0) & (road_mask == 1)
    # 모폴로지로 노이즈 제거
    lane = cv2.morphologyEx(lane.astype(np.uint8), cv2.MORPH_OPEN,
                            np.ones((3, 3), np.uint8))
    return lane > 0


# ============================================================
# STEP 1: Dark Zurich 변환
# ============================================================
DZ_GT_DIR = '/content/darkzurich/Dark_Zurich_val_anon/gt/val'
DZ_RGB_DIR = '/content/darkzurich/Dark_Zurich_val_anon/rgb_anon/val'

OUT_IMG = '/content/unified/images'
OUT_MASK = '/content/unified/masks'
os.makedirs(OUT_IMG, exist_ok=True)
os.makedirs(OUT_MASK, exist_ok=True)

dz_labels = sorted(glob.glob(f'{DZ_GT_DIR}/**/*labelTrainIds.png', recursive=True))
print(f"🌃 Dark Zurich 라벨 {len(dz_labels)}장 변환 시작")

dz_count = 0
for gt_path in tqdm(dz_labels, desc='Dark Zurich'):
    # 대응 RGB 찾기
    rel = os.path.relpath(gt_path, DZ_GT_DIR)
    rgb_name = rel.replace('_gt_labelTrainIds.png', '_rgb_anon.png')
    rgb_path = os.path.join(DZ_RGB_DIR, rgb_name)

    if not os.path.exists(rgb_path):
        continue

    # 1) 라벨 변환
    label = np.array(Image.open(gt_path))
    mask = cityscapes_to_ours4(label)

    # 2) 차선 추출 (도로 영역에서)
    rgb = np.array(Image.open(rgb_path).convert('RGB'))
    # 크기 맞추기 (라벨이 작을 수 있음)
    if rgb.shape[:2] != mask.shape:
        rgb_resized = cv2.resize(rgb, (mask.shape[1], mask.shape[0]))
    else:
        rgb_resized = rgb
    lane = extract_lane_marks(rgb_resized, mask)
    mask[lane] = 2  # Lane Mark 덮어쓰기

    # 3) 저장 (파일명 단순화)
    base = f'dz_{dz_count:04d}'
    Image.fromarray(rgb).save(f'{OUT_IMG}/{base}.png')
    Image.fromarray(mask).save(f'{OUT_MASK}/{base}.png')
    dz_count += 1

print(f"✅ Dark Zurich: {dz_count}장 변환 완료")


# ============================================================
# STEP 2: 기존 200장 재매핑 (6클래스 → 4클래스)
# ============================================================
# 원본 6클래스: 0=Undriv, 1=Road, 2=Lane, 3=MyBike, 4=Rider, 5=Moveable
# 새 4클래스:   0=Undriv, 1=Road, 2=Lane, 3=Background
# → My bike(3), Rider(4), Moveable(5)는 모두 255로 (YOLO가 처리)
OLD_IMG = '/content/data/images'           # 너 원본 경로에 맞게 수정 필요!
OLD_MASK = '/content/data/masks_generated'

remap_6to4 = {0: 0, 1: 1, 2: 2, 3: 255, 4: 255, 5: 255, 255: 3}

if os.path.exists(OLD_IMG) and os.path.exists(OLD_MASK):
    orig_masks = sorted(os.listdir(OLD_MASK))
    print(f"\n🏍️ 원본 200장 재매핑 시작")

    orig_count = 0
    for fname in tqdm(orig_masks, desc='Original'):
        if not fname.endswith('.png'):
            continue
        img_p = os.path.join(OLD_IMG, fname)
        mask_p = os.path.join(OLD_MASK, fname)
        if not os.path.exists(img_p):
            continue

        old_mask = np.array(Image.open(mask_p))
        new_mask = np.full_like(old_mask, 255, dtype=np.uint8)
        for old_id, new_id in remap_6to4.items():
            new_mask[old_mask == old_id] = new_id
        # 255(원본 배경)는 Background(3)로
        new_mask[old_mask == 255] = 3

        base = f'orig_{orig_count:04d}'
        shutil.copy(img_p, f'{OUT_IMG}/{base}.png')
        Image.fromarray(new_mask).save(f'{OUT_MASK}/{base}.png')
        orig_count += 1

    print(f"✅ 원본: {orig_count}장 재매핑 완료")
else:
    print(f"\n⚠️ 원본 경로 없음 — 아래 경로 확인/수정 필요:")
    print(f"   OLD_IMG = {OLD_IMG}")
    print(f"   OLD_MASK = {OLD_MASK}")
    print(f"   (Colab에서 원본 데이터 다시 다운로드 필요할 수 있음)")


# ============================================================
# STEP 3: 최종 확인
# ============================================================
total_img = len(os.listdir(OUT_IMG))
total_mask = len(os.listdir(OUT_MASK))
print(f"\n{'='*50}")
print(f"📦 통합 데이터셋 완성")
print(f"   이미지: {total_img}장")
print(f"   마스크: {total_mask}장")
print(f"   경로: /content/unified/")
print(f"{'='*50}")

# 샘플 시각화
import matplotlib.pyplot as plt
sample_files = sorted(os.listdir(OUT_IMG))[::max(1, total_img//3)][:3]
fig, axes = plt.subplots(len(sample_files), 2, figsize=(14, 5*len(sample_files)))
if len(sample_files) == 1:
    axes = [axes]
for i, f in enumerate(sample_files):
    img = np.array(Image.open(f'{OUT_IMG}/{f}'))
    mask = np.array(Image.open(f'{OUT_MASK}/{f}'))
    axes[i][0].imshow(img); axes[i][0].set_title(f'RGB: {f}'); axes[i][0].axis('off')
    axes[i][1].imshow(mask, cmap='tab10', vmin=0, vmax=9);
    axes[i][1].set_title(f'Mask (0=Und,1=Road,2=Lane,3=BG,255=ignore)'); axes[i][1].axis('off')
plt.tight_layout(); plt.show()

라벨-이미지 파일 쌍 매칭 검증

In [ ]:
# 실제로 쌍 매칭이 되는지 샘플 3개 테스트
import os

for gt_path in labels[:3]:
    rel = os.path.relpath(gt_path, '/content/darkzurich/Dark_Zurich_val_anon/gt/val')
    rgb_name = rel.replace('_gt_labelTrainIds.png', '_rgb_anon.png')
    rgb_path = os.path.join('/content/darkzurich/Dark_Zurich_val_anon/rgb_anon/val', rgb_name)
    exists = '✅' if os.path.exists(rgb_path) else '❌'
    print(f"{exists} GT:  {os.path.basename(gt_path)}")
    print(f"   RGB: {os.path.basename(rgb_path)}\n")

In [ ]:
# 원본 Kaggle 데이터가 Colab에 있는지
!ls /content/data/images 2>/dev/null | wc -l
!ls /content/data/masks_generated 2>/dev/null | wc -l

In [ ]:
# kaggle.json 업로드 먼저 (좌측 파일 탭에서 드래그해서 올리기)
!pip install kaggle -q
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d matthewinthehat/motorcyclenightride -p /content/
!unzip -q /content/motorcyclenightride.zip -d /content/data_raw/

# 심볼릭 링크로 경로 단순화
!ln -sf "/content/data_raw/www.acmeai.tech ODataset 1 - Motorcycle Night Ride Dataset" /content/data
!ls /content/data/images | head -3
!ls /content/data/masks_generated 2>/dev/null | wc -l

In [ ]:
# 업로드 확인
!ls -la kaggle.json

# 권한 세팅 + 다운로드
!pip install kaggle -q
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# 다운로드
!kaggle datasets download -d matthewinthehat/motorcyclenightride -p /content/
!unzip -q /content/motorcyclenightride.zip -d /content/data_raw/

# 심볼릭 링크 (따옴표 중요!)
!ln -sf "/content/data_raw/www.acmeai.tech ODataset 1 - Motorcycle Night Ride Dataset" /content/data

# 확인
!ls /content/data/images | head -3
!ls /content/data/masks_generated 2>/dev/null | wc -l

In [ ]:
# kaggle.json 구조 확인 (전체 키 출력은 안 되게)
import json
with open('kaggle.json') as f:
    data = json.load(f)
print(f"username: {data.get('username', '❌ 없음')}")
print(f"key 존재: {'✅' if data.get('key') else '❌'} ({len(data.get('key', ''))} chars)")


In [ ]:
!rm -rf ~/.kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets list -s motorcycle | head -5  # 작동 테스트


In [ ]:
!ls /content/data/
!ls /content/data/images 2>/dev/null | wc -l
!ls /content/data/masks_generated 2>/dev/null | wc -l

In [ ]:
# 드라이브에 올린 zip 위치 확인
!ls /content/drive/MyDrive/ | grep -i motor

# 압축 풀기 (파일명은 실제 것으로)
!unzip -q /content/drive/MyDrive/motorcycle_data.zip -d /content/

# 확인
!find /content -name "images" -type d | head -5
!find /content -name "masks_generated" -type d | head -5

Dark Zurich → 3클래스 마스크 전처리 (실제 사용된 버전)

In [ ]:
import os, glob, shutil
import numpy as np
from PIL import Image
from tqdm import tqdm

# 기존 unified 비우고 다시 시작
!rm -rf /content/unified/
os.makedirs('/content/unified/images', exist_ok=True)
os.makedirs('/content/unified/masks', exist_ok=True)

OUT_IMG = '/content/unified/images'
OUT_MASK = '/content/unified/masks'

# ============ Cityscapes 19 → 우리 3클래스 ============
CS_TO_OURS = {
    0: 1,    # road → Road(1)
    # 정적 장애물 → Undrivable(0)
    1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0,
    # 식물/하늘 → Background(2)
    8: 2, 9: 2, 10: 2,
    # 움직이는 객체 → ignore(255), YOLO 담당
    11: 255, 12: 255, 13: 255, 14: 255,
    15: 255, 16: 255, 17: 255, 18: 255,
}

def cs_to_ours3(label_np):
    out = np.full_like(label_np, 255, dtype=np.uint8)
    for cs_id, our_id in CS_TO_OURS.items():
        out[label_np == cs_id] = our_id
    return out


# ============ Dark Zurich 50장 변환 ============
DZ_GT = '/content/darkzurich/Dark_Zurich_val_anon/gt/val'
DZ_RGB = '/content/darkzurich/Dark_Zurich_val_anon/rgb_anon/val'

dz_labels = sorted(glob.glob(f'{DZ_GT}/**/*labelTrainIds.png', recursive=True))
print(f"🌃 Dark Zurich 라벨 {len(dz_labels)}장 변환")

count = 0
for gt_path in tqdm(dz_labels):
    rel = os.path.relpath(gt_path, DZ_GT)
    rgb_path = os.path.join(DZ_RGB, rel.replace('_gt_labelTrainIds.png', '_rgb_anon.png'))
    if not os.path.exists(rgb_path): continue

    label = np.array(Image.open(gt_path))
    mask = cs_to_ours3(label)
    rgb = np.array(Image.open(rgb_path).convert('RGB'))

    # 저장 (512x1024로 리사이즈해서 저장 — SegFormer 입력 크기에 맞춤)
    import cv2
    rgb_resized = cv2.resize(rgb, (1024, 512))
    mask_resized = cv2.resize(mask, (1024, 512), interpolation=cv2.INTER_NEAREST)

    base = f'dz_{count:04d}'
    Image.fromarray(rgb_resized).save(f'{OUT_IMG}/{base}.png')
    Image.fromarray(mask_resized).save(f'{OUT_MASK}/{base}.png')
    count += 1

print(f"\n✅ 총 {count}장 변환 완료")
print(f"   경로: /content/unified/\n")


# ============ 검증 시각화 ============
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 3클래스 + ignore 컬러맵
cmap = ListedColormap(['#808080', '#4444FF', '#44DD44'] + ['white']*252 + ['red'])

samples = sorted(os.listdir(OUT_IMG))[::max(1, count//4)][:4]
fig, axes = plt.subplots(len(samples), 2, figsize=(14, 5*len(samples)))

for i, f in enumerate(samples):
    img = np.array(Image.open(f'{OUT_IMG}/{f}'))
    mask = np.array(Image.open(f'{OUT_MASK}/{f}'))
    axes[i][0].imshow(img); axes[i][0].set_title(f'RGB: {f}'); axes[i][0].axis('off')
    axes[i][1].imshow(mask, cmap=cmap, vmin=0, vmax=255)
    axes[i][1].set_title('회:Und 파:Road 초:BG 빨:ignore')
    axes[i][1].axis('off')

plt.tight_layout(); plt.show()

# 클래스 분포 체크
from collections import Counter
all_pixels = Counter()
for f in os.listdir(OUT_MASK):
    mask = np.array(Image.open(f'{OUT_MASK}/{f}'))
    vals, cnts = np.unique(mask, return_counts=True)
    for v, c in zip(vals, cnts):
        all_pixels[int(v)] += int(c)

total = sum(all_pixels.values())
print(f"\n📊 픽셀 분포:")
labels = {0: 'Undrivable', 1: 'Road', 2: 'Background', 255: 'ignore(YOLO)'}
for k in sorted(all_pixels.keys()):
    ratio = all_pixels[k] / total * 100
    print(f"   {k:3d} {labels.get(k, '?'):15s}: {ratio:5.2f}%")

In [ ]:
# ============================================================
# 📦 SegFormer-B0 학습 스크립트 (Cityscapes pretrained → 3클래스)
# ============================================================
!pip install transformers -q

import os, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Device: {device}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

# ============ 설정 ============
CLASS_NAMES_3 = ['Undrivable', 'Road', 'Background']
NUM_CLASSES = 3
INPUT_H, INPUT_W = 512, 1024
BATCH_SIZE = 4     # T4면 4, A100이면 8
NUM_EPOCHS = 40
LR = 6e-5
IGNORE_INDEX = 255

IMG_DIR = '/content/unified/images'
MASK_DIR = '/content/unified/masks'

# ============ 강한 악조건 증강 (핵심!) ============
train_transform = A.Compose([
    A.Resize(height=INPUT_H, width=INPUT_W),

    # 악조건 시뮬레이션 (50% 확률)
    A.OneOf([
        A.RandomFog(fog_coef_lower=0.3, fog_coef_upper=0.7, alpha_coef=0.15, p=1.0),
        A.RandomRain(slant_lower=-15, slant_upper=15, drop_length=20,
                     blur_value=3, brightness_coefficient=0.7, p=1.0),
        A.MotionBlur(blur_limit=7, p=1.0),
        A.GaussNoise(var_limit=(20, 60), p=1.0),
    ], p=0.5),

    # 저조도 강화
    A.OneOf([
        A.RandomGamma(gamma_limit=(40, 100), p=1.0),
        A.RandomBrightnessContrast(brightness_limit=(-0.3, 0.1), contrast_limit=0.3, p=1.0),
    ], p=0.4),

    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(height=INPUT_H, width=INPUT_W),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ============ Dataset ============
class DarkZurichDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.files = file_list
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img = np.array(Image.open(f'{IMG_DIR}/{fname}').convert('RGB'))
        mask = np.array(Image.open(f'{MASK_DIR}/{fname}'))

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img = aug['image']
            mask = aug['mask'].long()
        return img, mask

# Split (90:10, 50장이라 val은 5장)
all_files = sorted(os.listdir(IMG_DIR))
np.random.seed(42)
shuffled = np.random.permutation(all_files)
split = int(len(shuffled) * 0.9)
train_files = shuffled[:split].tolist()
val_files = shuffled[split:].tolist()
print(f"📊 Train: {len(train_files)} | Val: {len(val_files)}")

train_ds = DarkZurichDataset(train_files, train_transform)
val_ds = DarkZurichDataset(val_files, val_transform)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ============ 모델: Cityscapes pretrained ============
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-cityscapes-512-1024",
    num_labels=NUM_CLASSES,
    id2label={i: n for i, n in enumerate(CLASS_NAMES_3)},
    label2id={n: i for i, n in enumerate(CLASS_NAMES_3)},
    ignore_mismatched_sizes=True,  # head 교체 시 필수
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"🧠 모델 파라미터: {total_params:,}")

# ============ Combined Loss (CE + Dice) ============
class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, pred, target):
        valid = (target != IGNORE_INDEX)
        pred_soft = F.softmax(pred, dim=1)
        target_safe = target.clone()
        target_safe[~valid] = 0
        target_oh = F.one_hot(target_safe, self.num_classes).permute(0, 3, 1, 2).float()
        target_oh = target_oh * valid.unsqueeze(1).float()

        inter = (pred_soft * target_oh).sum(dim=(2, 3))
        union = pred_soft.sum(dim=(2, 3)) + target_oh.sum(dim=(2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, num_classes, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
        self.dice = DiceLoss(num_classes)
        self.dw = dice_weight

    def forward(self, pred, target):
        # SegFormer 출력은 1/4 해상도 → 업샘플
        pred_up = F.interpolate(pred, size=target.shape[-2:],
                                mode='bilinear', align_corners=False)
        return (1 - self.dw) * self.ce(pred_up, target) + self.dw * self.dice(pred_up, target)

criterion = CombinedLoss(NUM_CLASSES, dice_weight=0.5)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# ============ IoU 계산 ============
def compute_iou(pred, target, num_classes=NUM_CLASSES):
    pred = pred.cpu().numpy()
    target = target.cpu().numpy()
    ious = []
    for c in range(num_classes):
        valid = (target != IGNORE_INDEX)
        pc = (pred == c) & valid
        tc = (target == c) & valid
        inter = (pc & tc).sum()
        union = (pc | tc).sum()
        ious.append(float('nan') if union == 0 else inter / union)
    return ious

# ============ 학습 루프 ============
best_miou = 0.0
history = {'train_loss': [], 'val_loss': [], 'mIoU': [], 'class_iou': []}

print(f"\n🚀 학습 시작 ({NUM_EPOCHS} epochs)\n")

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    train_loss_sum = 0; n_batches = 0
    pbar = tqdm(train_loader, desc=f'E{epoch} Train')
    for imgs, masks in pbar:
        imgs, masks = imgs.to(device), masks.to(device)
        out = model(pixel_values=imgs).logits
        loss = criterion(out, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item(); n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    train_loss = train_loss_sum / n_batches

    # Val
    model.eval()
    val_loss_sum = 0; vn = 0
    all_ious = [[] for _ in range(NUM_CLASSES)]
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            out = model(pixel_values=imgs).logits
            loss = criterion(out, masks)
            val_loss_sum += loss.item(); vn += 1

            pred_up = F.interpolate(out, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            preds = pred_up.argmax(dim=1)
            ious = compute_iou(preds, masks)
            for c, iou in enumerate(ious):
                if not np.isnan(iou): all_ious[c].append(iou)

    val_loss = val_loss_sum / vn
    class_ious = [np.mean(x) if x else 0.0 for x in all_ious]
    mIoU = np.mean(class_ious)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['mIoU'].append(mIoU)
    history['class_iou'].append(class_ious)

    print(f"\n📊 E{epoch}: train={train_loss:.4f} | val={val_loss:.4f} | mIoU={mIoU:.4f}")
    for i, (n_, iou) in enumerate(zip(CLASS_NAMES_3, class_ious)):
        print(f"     {i} {n_:12s}: {iou:.4f}")

    if mIoU > best_miou:
        best_miou = mIoU
        torch.save(model.state_dict(), '/content/best_segformer.pth')
        print(f"  💾 저장! (mIoU: {mIoU:.4f})")

print(f"\n🏆 최고 mIoU: {best_miou:.4f}")
print(f"   저장: /content/best_segformer.pth")

In [ ]:
# 새 셀에서 — 학습에 영향 없음
import shutil, os
os.makedirs('/content/drive/MyDrive/motorcycle_project/checkpoints', exist_ok=True)
shutil.copy('/content/best_segformer.pth',
            '/content/drive/MyDrive/motorcycle_project/checkpoints/best_segformer_mIoU_0.8735.pth')
print("✅ 드라이브 백업 완료")
!ls -lh /content/drive/MyDrive/motorcycle_project/checkpoints/

In [ ]:
# ================================================================
# STEP 10. 학습 결과 시각화 — 학습 곡선 + 클래스별 IoU
# ================================================================
import matplotlib.pyplot as plt
import numpy as np

epochs = range(1, len(history['train_loss']) + 1)
class_iou_arr = np.array(history['class_iou'])  # (epochs, 3)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ===== [1] Loss 곡선 =====
axes[0].plot(epochs, history['train_loss'], label='Train Loss',
             marker='o', color='#FF6B6B', linewidth=2, markersize=4)
axes[0].plot(epochs, history['val_loss'], label='Val Loss',
             marker='s', color='#4ECDC4', linewidth=2, markersize=4)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training / Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
# 중요 지점 표시
best_epoch = np.argmax(history['mIoU']) + 1
axes[0].axvline(x=best_epoch, color='gold', linestyle='--', alpha=0.7,
                label=f'Best Epoch: E{best_epoch}')

# ===== [2] mIoU 곡선 =====
axes[1].plot(epochs, history['mIoU'], marker='o', color='#2ECC71',
             linewidth=2.5, markersize=5)
axes[1].axhline(y=max(history['mIoU']), color='gold', linestyle='--',
                alpha=0.7, label=f'Best: {max(history["mIoU"]):.4f}')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('mIoU', fontsize=12)
axes[1].set_title('Validation mIoU', fontsize=14, fontweight='bold')
axes[1].fill_between(epochs, history['mIoU'], alpha=0.2, color='#2ECC71')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0.3, 0.95])

# ===== [3] 클래스별 IoU =====
colors_cls = ['#95A5A6', '#3498DB', '#27AE60']
class_names = ['Undrivable', 'Road', 'Background']
for i, (name, color) in enumerate(zip(class_names, colors_cls)):
    axes[2].plot(epochs, class_iou_arr[:, i], label=name,
                  marker='.', color=color, linewidth=2)
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('IoU per Class', fontsize=12)
axes[2].set_title('Per-Class IoU Over Epochs', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=11, loc='lower right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120, bbox_inches='tight')
# 드라이브에도 저장
import shutil, os
os.makedirs('/content/drive/MyDrive/motorcycle_project/results', exist_ok=True)
shutil.copy('/content/training_curves.png',
            '/content/drive/MyDrive/motorcycle_project/results/training_curves.png')
plt.show()
print("\n✅ 저장: training_curves.png (로컬 + 드라이브)")

In [ ]:
# ================================================================
# STEP 11. 최종 IoU 막대그래프 — 발표 슬라이드용 "한 장 요약"
# ================================================================
fig, ax = plt.subplots(figsize=(10, 6))

# E38 (최고) 기준 결과
best_idx = np.argmax(history['mIoU'])
final_ious = history['class_iou'][best_idx]
final_miou = history['mIoU'][best_idx]

classes = ['Undrivable', 'Road', 'Background', 'mIoU (avg)']
values = list(final_ious) + [final_miou]
colors = ['#95A5A6', '#3498DB', '#27AE60', '#E74C3C']

bars = ax.bar(classes, values, color=colors, edgecolor='black', linewidth=1.5)

# 막대 위에 수치 표시
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.015,
            f'{val:.4f}', ha='center', va='bottom',
            fontsize=13, fontweight='bold')

# 기준선 (0.80, 0.90)
ax.axhline(y=0.80, color='gray', linestyle=':', alpha=0.5, label='Excellent (0.80)')
ax.axhline(y=0.90, color='gold', linestyle=':', alpha=0.7, label='SOTA level (0.90)')

ax.set_ylim([0, 1.0])
ax.set_ylabel('IoU Score', fontsize=13)
ax.set_title(f'SegFormer-B0 Final Results (Best Epoch: E{best_idx+1})\n'
             f'Dark Zurich 야간 환경 | Cityscapes pretrained + 악조건 증강',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/final_scores.png', dpi=120, bbox_inches='tight')
shutil.copy('/content/final_scores.png',
            '/content/drive/MyDrive/motorcycle_project/results/final_scores.png')
plt.show()
print("\n✅ 저장: final_scores.png")

In [ ]:
# ================================================================
# STEP 12. 실제 Val 이미지 예측 시각화 — 원본 vs GT vs 예측
# ================================================================
from matplotlib.colors import ListedColormap
import torch.nn.functional as F

# 최고 모델 로드
model.load_state_dict(torch.load('/content/best_segformer.pth'))
model.eval()

# 컬러맵: 3클래스 + ignore
cmap = ListedColormap(['#808080', '#4444FF', '#44DD44'] + ['white']*252 + ['red'])

def denorm(img_tensor):
    """정규화 풀기"""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * std + mean
    return np.clip(img * 255, 0, 255).astype(np.uint8)

# Val 전체 예측해서 가장 잘 된 거 + 보통인 거 섞어서 5장
n_show = min(5, len(val_ds))
fig, axes = plt.subplots(n_show, 3, figsize=(18, 5*n_show))

model.eval()
with torch.no_grad():
    for i in range(n_show):
        img_t, mask_t = val_ds[i]
        img_b = img_t.unsqueeze(0).to(device)

        # 추론
        out = model(pixel_values=img_b).logits
        out_up = F.interpolate(out, size=mask_t.shape,
                                mode='bilinear', align_corners=False)
        pred = out_up.argmax(dim=1)[0].cpu().numpy()

        # 이 샘플의 mIoU
        sample_ious = compute_iou(
            out_up.argmax(dim=1), mask_t.unsqueeze(0).to(device), 3)
        valid_ious = [x for x in sample_ious if not np.isnan(x)]
        sample_miou = np.mean(valid_ious) if valid_ious else 0.0

        # 시각화
        img_vis = denorm(img_t)
        gt_vis = mask_t.numpy()

        axes[i][0].imshow(img_vis)
        axes[i][0].set_title(f'Input (Sample {i+1})', fontsize=11)
        axes[i][0].axis('off')

        axes[i][1].imshow(gt_vis, cmap=cmap, vmin=0, vmax=255)
        axes[i][1].set_title('Ground Truth', fontsize=11)
        axes[i][1].axis('off')

        axes[i][2].imshow(pred, cmap=cmap, vmin=0, vmax=255)
        axes[i][2].set_title(f'Prediction (mIoU: {sample_miou:.4f})',
                              fontsize=11, fontweight='bold', color='darkgreen')
        axes[i][2].axis('off')

plt.suptitle('Val Set Predictions — Gray: Undrivable | Blue: Road | Green: Background',
             fontsize=13, fontweight='bold', y=1.001)
plt.tight_layout()
plt.savefig('/content/predictions_sample.png', dpi=100, bbox_inches='tight')
shutil.copy('/content/predictions_sample.png',
            '/content/drive/MyDrive/motorcycle_project/results/predictions_sample.png')
plt.show()
print("\n✅ 저장: predictions_sample.png")

In [ ]:
# ================================================================
# STEP 13. 예측 결과 원본 위에 오버레이 — HUD 미리보기 느낌
# ================================================================
fig, axes = plt.subplots(3, 2, figsize=(16, 15))

with torch.no_grad():
    for row, i in enumerate([0, 2, 4][:min(3, len(val_ds))]):
        img_t, mask_t = val_ds[i]
        img_b = img_t.unsqueeze(0).to(device)

        out = model(pixel_values=img_b).logits
        out_up = F.interpolate(out, size=mask_t.shape,
                                mode='bilinear', align_corners=False)
        pred = out_up.argmax(dim=1)[0].cpu().numpy()

        img_vis = denorm(img_t)

        # 컬러 오버레이
        color_mask = np.zeros_like(img_vis)
        color_mask[pred == 0] = [128, 128, 128]   # Undrivable - 회색
        color_mask[pred == 1] = [50, 100, 255]     # Road - 파랑
        color_mask[pred == 2] = [50, 220, 50]      # Background - 초록

        # 반투명 합성
        overlay = (img_vis * 0.55 + color_mask * 0.45).astype(np.uint8)

        axes[row][0].imshow(img_vis)
        axes[row][0].set_title(f'Before: 원본 야간 이미지',
                                fontsize=12, fontweight='bold')
        axes[row][0].axis('off')

        axes[row][1].imshow(overlay)
        axes[row][1].set_title(f'After: SegFormer 예측 오버레이',
                                fontsize=12, fontweight='bold', color='darkblue')
        axes[row][1].axis('off')

plt.suptitle(
    f'🏍️ GUARDIAN — SegFormer-B0 Segmentation Results\n'
    f'Best mIoU: {max(history["mIoU"]):.4f} | Road IoU: 0.9011',
    fontsize=15, fontweight='bold', y=0.995
)
plt.tight_layout()
plt.savefig('/content/before_after.png', dpi=110, bbox_inches='tight')
shutil.copy('/content/before_after.png',
            '/content/drive/MyDrive/motorcycle_project/results/before_after.png')
plt.show()
print("\n✅ 저장: before_after.png")

In [ ]:
# ================================================================
# STEP 14. 데모 영상 생성 — SegFormer 예측 오버레이
# ================================================================
import os, cv2, numpy as np, torch, shutil
import torch.nn.functional as F
import imageio
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ----- 설정 -----
VIDEO_DIR = '/content/drive/MyDrive'
OUTPUT_DIR = '/content/drive/MyDrive/motorcycle_project/demo_videos'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 🎯 첫 영상: night_pov (학습 도메인 유사 → 성공 보장)
SOURCE_VIDEO = f'{VIDEO_DIR}/15285335_3840_2160_30fps.mp4'
ENV_LABEL = 'night_pov'

OUTPUT_W, OUTPUT_H = 1280, 720   # 4K → 720p
OUTPUT_FPS = 10
MAX_OUTPUT_FRAMES = 200           # 10fps × 20초 = 20초 영상

device = torch.device('cuda')

# ----- 모델 로드 (이미 있으면 재사용) -----
try:
    _ = model
    model.eval()
    print("✅ 모델 메모리에 있음")
except NameError:
    print("📥 드라이브에서 모델 재로드...")
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-cityscapes-512-1024",
        num_labels=3,
        id2label={0:'Undrivable', 1:'Road', 2:'Background'},
        label2id={'Undrivable':0, 'Road':1, 'Background':2},
        ignore_mismatched_sizes=True,
    ).to(device)
    model.load_state_dict(torch.load(
        '/content/drive/MyDrive/motorcycle_project/checkpoints/best_segformer_mIoU_0.8735.pth'))
    model.eval()
    print("✅ 로드 완료")

# ----- 전처리 -----
transform = A.Compose([
    A.Resize(512, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ----- 클래스 컬러 -----
COLORS = {
    0: (128, 128, 128),   # Undrivable
    1: (50, 100, 255),    # Road (파랑)
    2: (50, 220, 80),     # Background (초록)
}

def segment_frame(frame_rgb):
    aug = transform(image=frame_rgb)
    t = aug['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(pixel_values=t).logits
        out_up = F.interpolate(out, size=frame_rgb.shape[:2],
                               mode='bilinear', align_corners=False)
        mask = out_up.argmax(dim=1)[0].cpu().numpy()
    return mask

def render_overlay(frame, mask, alpha=0.45):
    color_mask = np.zeros_like(frame)
    for cls, color in COLORS.items():
        color_mask[mask == cls] = color
    return cv2.addWeighted(frame, 1-alpha, color_mask, alpha, 0)

def add_hud(frame, idx, total, env_label):
    H, W = frame.shape[:2]

    # 상단 바
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (W, 50), (15, 20, 30), -1)
    frame = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)
    cv2.putText(frame, f'GUARDIAN | {env_label.upper()}', (20, 33),
                cv2.FONT_HERSHEY_DUPLEX, 0.75, (0, 230, 255), 2)
    cv2.putText(frame, f'Frame {idx}/{total}', (W-180, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    # 하단 범례
    ly = H - 45
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, ly), (W, H), (15, 20, 30), -1)
    frame = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)

    x = 20
    for name, color in [('Road', (50,100,255)), ('Undrivable', (128,128,128)), ('Background', (50,220,80))]:
        cv2.rectangle(frame, (x, ly+15), (x+20, ly+35), color, -1)
        cv2.putText(frame, name, (x+25, ly+30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        x += 140

    cv2.putText(frame, 'SegFormer-B0 | mIoU: 0.8735', (W-300, ly+30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 230, 255), 1)
    return frame


# ----- 영상 확인 -----
if not os.path.exists(SOURCE_VIDEO):
    raise FileNotFoundError(f"❌ 영상 없음: {SOURCE_VIDEO}\n   드라이브 경로 확인 필요")

print(f"\n🎬 원본: {os.path.basename(SOURCE_VIDEO)}")
reader = imageio.get_reader(SOURCE_VIDEO)
meta = reader.get_meta_data()
src_fps = meta.get('fps', 30)
print(f"   원본 FPS: {src_fps:.1f}")

# 30fps → 10fps 다운샘플 (매 3프레임마다 1개)
sample_step = max(1, int(round(src_fps / OUTPUT_FPS)))
indices = list(range(0, MAX_OUTPUT_FRAMES * sample_step, sample_step))
print(f"   샘플 간격: {sample_step} → 출력 {OUTPUT_FPS}fps, {MAX_OUTPUT_FRAMES}프레임 ({MAX_OUTPUT_FRAMES/OUTPUT_FPS:.0f}초)\n")

# ----- 프레임별 처리 -----
output_frames = []
for i, idx in enumerate(tqdm(indices, desc='Rendering')):
    try:
        reader.set_image_index(idx)
        frame = reader.get_next_data()
    except (IndexError, Exception):
        print(f"\n⚠️ 프레임 {idx}에서 끝남")
        break

    frame = cv2.resize(frame, (OUTPUT_W, OUTPUT_H))
    mask = segment_frame(frame)
    overlay = render_overlay(frame, mask)
    hud = add_hud(overlay, i+1, len(indices), ENV_LABEL)
    output_frames.append(hud)

reader.close()

# ----- 저장 -----
output_path = f'{OUTPUT_DIR}/demo_{ENV_LABEL}.mp4'
print(f"\n💾 저장 중... ({len(output_frames)} frames)")
imageio.mimsave(output_path, output_frames, fps=OUTPUT_FPS, codec='libx264')

print(f"\n✅ 완성!")
print(f"   📁 파일: {output_path}")
print(f"   ⏱️ 길이: {len(output_frames)/OUTPUT_FPS:.1f}초")
print(f"   📐 해상도: {OUTPUT_W}x{OUTPUT_H}")

# 재생
from IPython.display import Video
Video(output_path, embed=True, width=960)

In [ ]:
import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
!nvidia-smi | head -5

In [ ]:
# 1) 드라이브 먼저
from google.colab import drive
drive.mount('/content/drive')

# 2) transformers (Colab에 이미 있을 수도 있음 — 있으면 스킵됨)
!pip install transformers -q

# 3) 모델 로드
import torch
from transformers import SegformerForSemanticSegmentation

device = torch.device('cuda')

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-cityscapes-512-1024",
    num_labels=3,
    id2label={0:'Undrivable', 1:'Road', 2:'Background'},
    label2id={'Undrivable':0, 'Road':1, 'Background':2},
    ignore_mismatched_sizes=True,
).to(device)

model.load_state_dict(torch.load(
    '/content/drive/MyDrive/motorcycle_project/checkpoints/best_segformer_mIoU_0.8735.pth'))
model.eval()
print("✅ 모델 로드 완료")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!ls /content/drive/MyDrive/motorcycle_project/checkpoints/

In [ ]:
import torch
from transformers import SegformerForSemanticSegmentation

device = torch.device('cuda')

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-cityscapes-512-1024",
    num_labels=3,
    id2label={0:'Undrivable', 1:'Road', 2:'Background'},
    label2id={'Undrivable':0, 'Road':1, 'Background':2},
    ignore_mismatched_sizes=True,
).to(device)

model.load_state_dict(torch.load(
    '/content/drive/MyDrive/motorcycle_project/checkpoints/best_segformer_mIoU_0.8735.pth'))
model.eval()
print("✅ 모델 로드 완료 — 영상 생성 준비됐음")

In [ ]:
# 우리 학습 가중치가 제대로 덮어쓰였는지 확인
import torch

# 방금 만든 model에 우리 체크포인트 확실히 로드
ckpt_path = '/content/drive/MyDrive/motorcycle_project/checkpoints/best_segformer_mIoU_0.8735.pth'
state_dict = torch.load(ckpt_path, map_location='cuda')

# 로드 + 검증
result = model.load_state_dict(state_dict, strict=False)
print(f"Missing keys: {len(result.missing_keys)}")      # 0이어야 정상
print(f"Unexpected keys: {len(result.unexpected_keys)}") # 0이어야 정상
print(f"\n✅ 우리 학습 가중치 로드 완료 (mIoU 0.8735)")
model.eval()

segformer 동영상


In [ ]:
# ================================================================
# STEP 14. 데모 영상 생성 — SegFormer 예측 오버레이
# ================================================================
import os, cv2, numpy as np, torch, shutil
import torch.nn.functional as F
import imageio
from tqdm import tqdm
from transformers import SegformerForSemanticSegmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ----- 설정 -----
VIDEO_DIR = '/content/drive/MyDrive'
OUTPUT_DIR = '/content/drive/MyDrive/motorcycle_project/demo_videos'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 🎯 첫 영상: night_pov (학습 도메인 유사 → 성공 보장)
SOURCE_VIDEO = f'{VIDEO_DIR}/15285335_3840_2160_30fps.mp4'
ENV_LABEL = 'night_pov'

OUTPUT_W, OUTPUT_H = 1280, 720   # 4K → 720p
OUTPUT_FPS = 10
MAX_OUTPUT_FRAMES = 200           # 10fps × 20초 = 20초 영상

device = torch.device('cuda')

# ----- 모델 로드 (이미 있으면 재사용) -----
try:
    _ = model
    model.eval()
    print("✅ 모델 메모리에 있음")
except NameError:
    print("📥 드라이브에서 모델 재로드...")
    model = SegformerForSemanticSegmentation.from_pretrained(
        "nvidia/segformer-b0-finetuned-cityscapes-512-1024",
        num_labels=3,
        id2label={0:'Undrivable', 1:'Road', 2:'Background'},
        label2id={'Undrivable':0, 'Road':1, 'Background':2},
        ignore_mismatched_sizes=True,
    ).to(device)
    model.load_state_dict(torch.load(
        '/content/drive/MyDrive/motorcycle_project/checkpoints/best_segformer_mIoU_0.8735.pth'))
    model.eval()
    print("✅ 로드 완료")

# ----- 전처리 -----
transform = A.Compose([
    A.Resize(512, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ----- 클래스 컬러 -----
COLORS = {
    0: (128, 128, 128),   # Undrivable
    1: (50, 100, 255),    # Road (파랑)
    2: (50, 220, 80),     # Background (초록)
}

def segment_frame(frame_rgb):
    aug = transform(image=frame_rgb)
    t = aug['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(pixel_values=t).logits
        out_up = F.interpolate(out, size=frame_rgb.shape[:2],
                               mode='bilinear', align_corners=False)
        mask = out_up.argmax(dim=1)[0].cpu().numpy()
    return mask

def render_overlay(frame, mask, alpha=0.45):
    color_mask = np.zeros_like(frame)
    for cls, color in COLORS.items():
        color_mask[mask == cls] = color
    return cv2.addWeighted(frame, 1-alpha, color_mask, alpha, 0)

def add_hud(frame, idx, total, env_label):
    H, W = frame.shape[:2]

    # 상단 바
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (W, 50), (15, 20, 30), -1)
    frame = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)
    cv2.putText(frame, f'GUARDIAN | {env_label.upper()}', (20, 33),
                cv2.FONT_HERSHEY_DUPLEX, 0.75, (0, 230, 255), 2)
    cv2.putText(frame, f'Frame {idx}/{total}', (W-180, 32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

    # 하단 범례
    ly = H - 45
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, ly), (W, H), (15, 20, 30), -1)
    frame = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)

    x = 20
    for name, color in [('Road', (50,100,255)), ('Undrivable', (128,128,128)), ('Background', (50,220,80))]:
        cv2.rectangle(frame, (x, ly+15), (x+20, ly+35), color, -1)
        cv2.putText(frame, name, (x+25, ly+30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        x += 140

    cv2.putText(frame, 'SegFormer-B0 | mIoU: 0.8735', (W-300, ly+30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 230, 255), 1)
    return frame


# ----- 영상 확인 -----
if not os.path.exists(SOURCE_VIDEO):
    raise FileNotFoundError(f"❌ 영상 없음: {SOURCE_VIDEO}\n   드라이브 경로 확인 필요")

print(f"\n🎬 원본: {os.path.basename(SOURCE_VIDEO)}")
reader = imageio.get_reader(SOURCE_VIDEO)
meta = reader.get_meta_data()
src_fps = meta.get('fps', 30)
print(f"   원본 FPS: {src_fps:.1f}")

# 30fps → 10fps 다운샘플 (매 3프레임마다 1개)
sample_step = max(1, int(round(src_fps / OUTPUT_FPS)))
indices = list(range(0, MAX_OUTPUT_FRAMES * sample_step, sample_step))
print(f"   샘플 간격: {sample_step} → 출력 {OUTPUT_FPS}fps, {MAX_OUTPUT_FRAMES}프레임 ({MAX_OUTPUT_FRAMES/OUTPUT_FPS:.0f}초)\n")

# ----- 프레임별 처리 -----
output_frames = []
for i, idx in enumerate(tqdm(indices, desc='Rendering')):
    try:
        reader.set_image_index(idx)
        frame = reader.get_next_data()
    except (IndexError, Exception):
        print(f"\n⚠️ 프레임 {idx}에서 끝남")
        break

    frame = cv2.resize(frame, (OUTPUT_W, OUTPUT_H))
    mask = segment_frame(frame)
    overlay = render_overlay(frame, mask)
    hud = add_hud(overlay, i+1, len(indices), ENV_LABEL)
    output_frames.append(hud)

reader.close()

# ----- 저장 -----
output_path = f'{OUTPUT_DIR}/demo_{ENV_LABEL}.mp4'
print(f"\n💾 저장 중... ({len(output_frames)} frames)")
imageio.mimsave(output_path, output_frames, fps=OUTPUT_FPS, codec='libx264')

print(f"\n✅ 완성!")
print(f"   📁 파일: {output_path}")
print(f"   ⏱️ 길이: {len(output_frames)/OUTPUT_FPS:.1f}초")
print(f"   📐 해상도: {OUTPUT_W}x{OUTPUT_H}")

# 재생
from IPython.display import Video
Video(output_path, embed=True, width=960)

In [ ]:
# ================================================================
# 🎯 YOLOv8 설치 (torch 건드리지 않는 안전 버전)
# ================================================================

# --deps 빼고 ultralytics만 설치 (의존성 자동 업그레이드 금지!)
!pip install ultralytics --no-deps -q

# 필요한 의존성 몇 개만 개별 설치 (torch 빼고)
!pip install "opencv-python>=4.6" "pyyaml>=5.3" "requests>=2.23" "pillow>=7.1.2" "pandas>=1.1.4" "seaborn>=0.11.0" "thop>=0.1.1" "tqdm>=4.64.0" --upgrade -q

# 설치 확인
import ultralytics
print(f"✅ Ultralytics: {ultralytics.__version__}")

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")  # ← 여전히 True여야 함!

In [ ]:
from ultralytics import YOLO

# YOLOv8n-seg nano (6MB 다운로드)
yolo_model = YOLO('yolov8n-seg.pt')

# 우리가 쓸 클래스
TARGET_CLASSES = {
    0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle',
    5: 'bus', 7: 'truck'
}

print(f"✅ YOLO 로드 완료")
print(f"   우리가 쓸 클래스: {list(TARGET_CLASSES.values())}")

In [ ]:
# ================================================================
# STEP 15. SegFormer + YOLO 통합 영상 생성
# ================================================================
import os, cv2, numpy as np, torch, imageio
import torch.nn.functional as F
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ----- 설정 -----
SOURCE_VIDEO = '/content/drive/MyDrive/15285335_3840_2160_30fps.mp4'
ENV_LABEL = 'night_pov'
OUTPUT_DIR = '/content/drive/MyDrive/motorcycle_project/demo_videos'
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_W, OUTPUT_H = 1280, 720
OUTPUT_FPS = 10
MAX_OUTPUT_FRAMES = 200
device = torch.device('cuda')

# ----- SegFormer 전처리 -----
seg_transform = A.Compose([
    A.Resize(512, 1024),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ----- 컬러 팔레트 -----
SEG_COLORS = {
    0: (128, 128, 128),   # Undrivable
    1: (50, 100, 255),     # Road
    2: (50, 220, 80),      # Background
}

# YOLO 클래스별 박스 색
YOLO_COLORS = {
    'person':     (255, 80, 80),      # 빨강 — 제일 중요
    'bicycle':    (255, 180, 50),
    'car':        (50, 200, 255),      # 시안 — 가장 흔함
    'motorcycle': (255, 100, 200),    # 핑크 — 동료 바이크
    'bus':        (200, 100, 255),
    'truck':      (255, 220, 50),      # 노랑
}

def segment_frame(frame_rgb):
    t = seg_transform(image=frame_rgb)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(pixel_values=t).logits
        out_up = F.interpolate(out, size=frame_rgb.shape[:2],
                                mode='bilinear', align_corners=False)
        return out_up.argmax(dim=1)[0].cpu().numpy()


def draw_yolo_boxes(frame, yolo_result):
    """YOLO 결과 → 박스 + 라벨 + ID 그리기"""
    if yolo_result.boxes is None or len(yolo_result.boxes) == 0:
        return frame, 0

    count = 0
    for box in yolo_result.boxes:
        cls_id = int(box.cls[0])
        cls_name = yolo_model.names[cls_id]
        if cls_name not in YOLO_COLORS:
            continue

        color = YOLO_COLORS[cls_name]
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])

        # 추적 ID
        tid = int(box.id[0]) if box.id is not None else -1

        # 박스
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        # 라벨 배경
        label = f"{cls_name}#{tid}" if tid > 0 else cls_name
        label += f" {conf:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_DUPLEX, 0.5, 1)
        cv2.rectangle(frame, (x1, y1-th-8), (x1+tw+6, y1), color, -1)
        cv2.putText(frame, label, (x1+3, y1-4),
                    cv2.FONT_HERSHEY_DUPLEX, 0.5, (255, 255, 255), 1)
        count += 1
    return frame, count


def render_segmentation(frame, mask, alpha=0.4):
    color_mask = np.zeros_like(frame)
    for cls, color in SEG_COLORS.items():
        color_mask[mask == cls] = color
    return cv2.addWeighted(frame, 1-alpha, color_mask, alpha, 0)


def add_hud(frame, idx, total, env_label, obj_count):
    H, W = frame.shape[:2]

    # 상단 바
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (W, 55), (15, 20, 30), -1)
    frame = cv2.addWeighted(overlay, 0.75, frame, 0.25, 0)
    cv2.putText(frame, f'GUARDIAN | {env_label.upper()}', (20, 35),
                cv2.FONT_HERSHEY_DUPLEX, 0.8, (0, 230, 255), 2)
    cv2.putText(frame, f'Objects: {obj_count}', (W//2-50, 35),
                cv2.FONT_HERSHEY_DUPLEX, 0.7, (255, 200, 50), 2)
    cv2.putText(frame, f'{idx}/{total}', (W-100, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    # 하단 범례
    ly = H - 50
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, ly), (W, H), (15, 20, 30), -1)
    frame = cv2.addWeighted(overlay, 0.75, frame, 0.25, 0)

    # Seg 범례
    x = 20
    cv2.putText(frame, 'SEG:', (x, ly+30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)
    x += 45
    for name, color in [('Road', SEG_COLORS[1]), ('Und', SEG_COLORS[0]), ('BG', SEG_COLORS[2])]:
        cv2.rectangle(frame, (x, ly+18), (x+15, ly+35), color, -1)
        cv2.putText(frame, name, (x+20, ly+32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
        x += 75

    # YOLO 정보
    cv2.putText(frame, 'DET: YOLOv8 + ByteTrack', (W-280, ly+32),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 230, 255), 1)
    return frame


# ----- 영상 읽기 -----
reader = imageio.get_reader(SOURCE_VIDEO)
src_fps = reader.get_meta_data().get('fps', 30)
sample_step = max(1, int(round(src_fps / OUTPUT_FPS)))
indices = list(range(0, MAX_OUTPUT_FRAMES * sample_step, sample_step))
print(f"🎬 원본 {src_fps:.0f}fps → 출력 {OUTPUT_FPS}fps, {MAX_OUTPUT_FRAMES}프레임 ({MAX_OUTPUT_FRAMES/OUTPUT_FPS:.0f}초)\n")

# ----- 프레임별 처리 -----
output_frames = []
total_detections = 0

for i, idx in enumerate(tqdm(indices, desc='Rendering')):
    try:
        reader.set_image_index(idx)
        frame = reader.get_next_data()
    except:
        break

    frame = cv2.resize(frame, (OUTPUT_W, OUTPUT_H))

    # SegFormer 추론
    seg_mask = segment_frame(frame)

    # YOLO 추적 (persist=True로 ID 유지)
    yolo_r = yolo_model.track(frame, persist=True, tracker='bytetrack.yaml',
                               conf=0.35, verbose=False)[0]

    # 1) Seg 오버레이
    frame = render_segmentation(frame, seg_mask, alpha=0.4)

    # 2) YOLO 박스
    frame, obj_count = draw_yolo_boxes(frame, yolo_r)
    total_detections += obj_count

    # 3) HUD
    frame = add_hud(frame, i+1, len(indices), ENV_LABEL, obj_count)

    output_frames.append(frame)

reader.close()

# ----- 저장 -----
output_path = f'{OUTPUT_DIR}/demo_{ENV_LABEL}_v2.mp4'
print(f"\n💾 저장 중... ({len(output_frames)} frames)")
imageio.mimsave(output_path, output_frames, fps=OUTPUT_FPS, codec='libx264')

print(f"\n✅ 완성!")
print(f"   📁 {output_path}")
print(f"   ⏱️ {len(output_frames)/OUTPUT_FPS:.1f}초")
print(f"   🎯 총 감지: {total_detections}개 객체 (평균 {total_detections/len(output_frames):.1f}/frame)")

from IPython.display import Video
Video(output_path, embed=True, width=960)

In [ ]:
# ================================================================
# STEP 16. 나머지 4개 환경 영상 일괄 생성 (배치)
# ================================================================
import os, imageio, cv2, numpy as np
from tqdm import tqdm

# ----- 영상별 환경 라벨 매핑 -----
VIDEO_MAPPING = {
    '12884761_2160_3840_25fps.mp4': 'fog',           # 🌫️ 안개
    '14537338-uhd_3840_2160_30fps.mp4': 'dusk',      # 🌅 석양
    '14964000_2160_3840_30fps.mp4': 'night_rain',    # 🌧️ 비 야간
    '15300886_3840_2160_60fps.mp4': 'night_city',    # 🏙️ 야간 도시
}

VIDEO_DIR = '/content/drive/MyDrive'
OUTPUT_DIR = '/content/drive/MyDrive/motorcycle_project/demo_videos'


def process_video(source_path, env_label,
                  out_w=1280, out_h=720,
                  out_fps=10, max_frames=200):
    """한 영상 처리 — 위에서 정의한 segment_frame, draw_yolo_boxes,
       render_segmentation, add_hud 함수 재사용"""

    if not os.path.exists(source_path):
        print(f"❌ 없음: {source_path}")
        return None

    print(f"\n{'='*60}")
    print(f"🎬 {env_label.upper()} — {os.path.basename(source_path)}")
    print(f"{'='*60}")

    reader = imageio.get_reader(source_path)
    src_fps = reader.get_meta_data().get('fps', 30)
    sample_step = max(1, int(round(src_fps / out_fps)))
    indices = list(range(0, max_frames * sample_step, sample_step))

    frames = []
    total_det = 0

    for i, idx in enumerate(tqdm(indices, desc=env_label)):
        try:
            reader.set_image_index(idx)
            frame = reader.get_next_data()
        except:
            break

        frame = cv2.resize(frame, (out_w, out_h))

        # SegFormer
        seg_mask = segment_frame(frame)

        # YOLO (새 영상이니 tracker 초기화 — persist=False 첫 프레임, 이후 True)
        yolo_r = yolo_model.track(
            frame,
            persist=(i > 0),   # 첫 프레임만 reset
            tracker='bytetrack.yaml',
            conf=0.35,
            verbose=False
        )[0]

        # 렌더링
        frame = render_segmentation(frame, seg_mask, alpha=0.4)
        frame, obj_count = draw_yolo_boxes(frame, yolo_r)
        frame = add_hud(frame, i+1, len(indices), env_label, obj_count)

        frames.append(frame)
        total_det += obj_count

    reader.close()

    # 저장
    output_path = f'{OUTPUT_DIR}/demo_{env_label}_v2.mp4'
    imageio.mimsave(output_path, frames, fps=out_fps, codec='libx264')

    print(f"✅ {output_path}")
    print(f"   ⏱️ {len(frames)/out_fps:.1f}초 | 🎯 평균 {total_det/max(1,len(frames)):.1f}개/frame")

    return output_path


# ----- 4개 순차 처리 -----
os.makedirs(OUTPUT_DIR, exist_ok=True)
outputs = {}

for video_name, env_label in VIDEO_MAPPING.items():
    source = f'{VIDEO_DIR}/{video_name}'
    result = process_video(source, env_label)
    if result:
        outputs[env_label] = result


# ----- 최종 결과 정리 -----
print(f"\n{'='*60}")
print(f"🎉 완료! 총 {len(outputs)}개 영상 생성")
print(f"{'='*60}\n")
for env, path in outputs.items():
    size_mb = os.path.getsize(path) / (1024*1024)
    print(f"  {env:12s} → {os.path.basename(path)} ({size_mb:.1f} MB)")

print(f"\n📁 저장 경로: {OUTPUT_DIR}")

In [ ]:
from IPython.display import Video, display, HTML

for env, path in outputs.items():
    display(HTML(f'<h3>🎬 {env.upper()}</h3>'))
    display(Video(path, embed=True, width=800))
    display(HTML('<hr>'))

In [ ]:
# ================================================================
# 🧹 GitHub 업로드용 노트북 정리 (widgets 메타데이터 제거)
# ================================================================
import json, os

# 본인 노트북 경로 — 지금 열린 노트북 이름 확인해서 바꿔
NOTEBOOK_NAME = 'Untitled0.ipynb'  # ← 필요시 수정

# Colab의 노트북 원본 경로
src = f'/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}'

# 대체로 이 경로에 있음
if not os.path.exists(src):
    # 백업으로 현재 실행 중인 노트북 검색
    import glob
    found = glob.glob('/content/drive/MyDrive/**/*.ipynb', recursive=True)
    print("❌ 직접 경로 못 찾음. 노트북 후보:")
    for f in found[:10]:
        print(f"   {f}")
    raise FileNotFoundError("위 목록에서 실제 경로 복사해서 NOTEBOOK_NAME 수정")

# 읽기
with open(src, 'r', encoding='utf-8') as f:
    nb = json.load(f)

# 1) widgets 메타데이터 제거 (GitHub 렌더링 실패 원인)
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']
    print("✅ widgets 메타데이터 제거")

# 2) 각 셀의 execution count/outputs 정리 (옵션 — 파일 크기 확 줄어듦)
CLEAR_OUTPUTS = False  # True로 바꾸면 모든 출력 지움 (용량↓)

if CLEAR_OUTPUTS:
    for cell in nb['cells']:
        if cell['cell_type'] == 'code':
            cell['outputs'] = []
            cell['execution_count'] = None

# 저장 — 같은 드라이브 폴더에 _clean 버전으로
dst = src.replace('.ipynb', '_for_github.ipynb')
with open(dst, 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print(f"\n✅ 정리 완료: {dst}")
print(f"   원본 크기: {os.path.getsize(src)/(1024*1024):.1f} MB")
print(f"   정리본: {os.path.getsize(dst)/(1024*1024):.1f} MB")
print(f"\n이 _for_github.ipynb 파일을 GitHub에 업로드하면 됨")